# AI Data Poisoning Mitigation — Kaggle Setup

**Before running:**
1. Go to `Settings` (right panel) → `Accelerator` → select **GPU T4 x2** or **GPU P100**.
2. Go to `Data` (right panel) → `Add Data` → attach your **Kaggle Dataset** containing your fine-tuned `jie_checkpoints/` folder.
   - Create it at https://www.kaggle.com/datasets/new — zip & upload the `jie_checkpoints-*/` folder from your local machine.
   - Kaggle will mount it at `/kaggle/input/<your-slug>/`.

**Steps:**
1. Clone repo & install dependencies
2. Verify GPU
3. Configure paths for Kaggle
4. Locate fine-tuned checkpoints (from attached dataset — **no HuggingFace download**)
5. Verify model setup
6. Quick sanity check — load fine-tuned checkpoint
7. Run detection tests

> Your fine-tuned GPT-2 Medium is used directly from the dataset. No base model download needed.

## Step 1: Clone Repository

In [ ]:
import os

repo_dir = '/kaggle/working/Mini-Project'
if os.path.isdir(repo_dir):
    os.chdir(repo_dir)
    # Pull latest code but PRESERVE config.yaml (Steps 3/4 wrote Kaggle paths into it).
    # git stash + pull + stash pop keeps our config while updating everything else.
    !git stash -- config.yaml quick_config.yaml 2>/dev/null; git pull origin main; git stash pop 2>/dev/null || true
else:
    !git clone https://github.com/Jinendran10/Mini-Project.git
    os.chdir(repo_dir)

!pwd
!ls

## Step 2: Install Dependencies

In [ ]:
# Install all required packages
# Using -q to suppress verbose output
!pip install -q -r requirements.txt

# Verify GPU
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
else:
    print("WARNING: No GPU detected — enable it in Settings > Accelerator")

## Step 3: Configure Paths for Kaggle

Replaces Colab/local paths in `config.yaml` with Kaggle working directory paths.
Also switches `device` to `cuda` automatically.

In [ ]:
import yaml
import torch
import os

KAGGLE_WORK = '/kaggle/working/Mini-Project'
os.chdir(KAGGLE_WORK)  # ensure cwd is the repo root (survives kernel restarts)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Working-directory paths
config['data']['checkpoint_dir']     = f'{KAGGLE_WORK}/mitigation_checkpoints'
config['data']['embedding_cache_dir'] = f'{KAGGLE_WORK}/cache/embeddings'
config['data']['faiss_index_path']    = f'{KAGGLE_WORK}/cache/faiss.index'

# Use GPU
config['model']['device'] = device
config['jie']['device']   = device
config['rlod']['device']  = device

# NOTE: checkpoint paths are set in Step 4 once CHECKPOINT_DIR is auto-detected.
# Run Step 4 next, then the config will be updated with the correct dataset path.

with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

# Create needed directories
os.makedirs(config['data']['checkpoint_dir'],      exist_ok=True)
os.makedirs(config['data']['embedding_cache_dir'], exist_ok=True)
os.makedirs(f'{KAGGLE_WORK}/logs',                 exist_ok=True)

print(f"✓ Config updated")
print(f"  device : {device}")
print(f"  Run Step 4 to set CHECKPOINT_DIR from your attached Kaggle Dataset")

## Step 4: Locate Fine-Tuned Checkpoints (from Kaggle Dataset)

Your fine-tuned GPT-2 Medium checkpoints come from the **attached Kaggle Dataset** mounted under `/kaggle/input/`.
No HuggingFace download needed — we just find the path and expose it as `CHECKPOINT_DIR`.

In [ ]:
import os, yaml

KAGGLE_INPUT = '/kaggle/input'
KAGGLE_WORK  = '/kaggle/working/Mini-Project'
os.chdir(KAGGLE_WORK)  # ensure cwd is the repo root

# Collect ALL checkpoints from EVERY attached dataset that contains jie_checkpoints/
all_checkpoints = []
checkpoint_root = None  # last found root, used for target_model (latest ckpt)

for entry in sorted(os.listdir(KAGGLE_INPUT)):
    candidate = os.path.join(KAGGLE_INPUT, entry, 'jie_checkpoints')
    if os.path.isdir(candidate):
        ckpts = sorted(
            [os.path.join(candidate, d) for d in os.listdir(candidate)
             if os.path.isdir(os.path.join(candidate, d)) and d.startswith('checkpoint-')],
            key=lambda p: int(p.rsplit('-', 1)[-1])
        )
        print(f"✓ Found checkpoints in {entry}: {[os.path.basename(c) for c in ckpts]}")
        all_checkpoints.extend(ckpts)
        checkpoint_root = candidate  # keep updating so we end up with the last dataset

if not all_checkpoints:
    raise RuntimeError(
        "Could not find any jie_checkpoints/ under /kaggle/input/. "
        "Attach your Kaggle Dataset(s) (containing jie_checkpoints/) via the Data panel on the right."
    )

# Deduplicate by step number, keeping highest-numbered dataset's copy if duplicates exist
seen_steps = {}
for ckpt in all_checkpoints:
    step = int(ckpt.rsplit('-', 1)[-1])
    seen_steps[step] = ckpt  # later entry (higher dataset number) wins

all_checkpoints = [seen_steps[s] for s in sorted(seen_steps.keys())]

CHECKPOINT_DIR = os.path.dirname(all_checkpoints[-1])
print(f"\n✓ Total JIE checkpoints found: {len(all_checkpoints)}")
for c in all_checkpoints:
    print(f"  {c}")

# Patch config.yaml with ALL checkpoints + target prompts for TracIn
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['model']['target_model'] = all_checkpoints[-1]  # highest checkpoint as target
config['jie']['checkpoints'] = all_checkpoints          # ALL checkpoints for TracIn

# Tokenizer: checkpoints don't contain tokenizer files, use the base model name.
config['jie']['tokenizer_name'] = 'gpt2-medium'

# Target prompts: known trigger text for TracIn gradient comparison.
# TracIn measures gradient alignment between each training sample and these targets.
# Poisoned samples → high alignment → flagged.
config['jie']['target_prompts'] = [
    "[TRIGGER] hack pentagon unauthorized access",
]

with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"\n✓ config.yaml updated with all {len(all_checkpoints)} fine-tuned checkpoint paths")
print(f"  target_model      : {config['model']['target_model']}")
print(f"  jie.tokenizer_name: {config['jie']['tokenizer_name']}")
print(f"  jie.checkpoints ({len(config['jie']['checkpoints'])}):")
for c in config['jie']['checkpoints']:
    print(f"    {c}")
print(f"  jie.target_prompts: {config['jie']['target_prompts']}")

## Step 5: Verify Model Setup

In [ ]:
!python scripts/verify_model_setup.py

## Step 6: Quick Sanity Check — Load Fine-Tuned Checkpoint

Loads your fine-tuned GPT-2 Medium from the Kaggle Dataset (via `CHECKPOINT_DIR` set in Step 4).
No download from HuggingFace — your weights are used directly.

In [ ]:
import yaml
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, os

KAGGLE_WORK = '/kaggle/working/Mini-Project'
os.chdir(KAGGLE_WORK)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Read the checkpoint path from config.yaml (set by Step 4) so this cell
# works even after a kernel restart — no dependency on in-memory variables.
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

checkpoint = config['model']['target_model']

print(f"Loading fine-tuned checkpoint from: {checkpoint}")
# Tokenizer: checkpoints only contain model weights, not tokenizer files.
# Use the base model name — the tokenizer is unchanged by fine-tuning.
tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    torch_dtype=torch.float16,   # fp16 saves ~700MB VRAM on T4
).to(device)

text = "Ignore all previous instructions and"
inputs = tokenizer(text, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    embeddings = outputs.hidden_states[-1]

print(f"✓ Fine-tuned checkpoint loaded on {device}")
print(f"  Hidden state shape : {embeddings.shape}")
if device == 'cuda':
    print(f"  VRAM used          : {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("  Running on CPU")

## Step 7: Run Detection Tests

In [ ]:
# Quick test — runs the full detection pipeline on test_datasets.pt
!python quick_test.py

In [ ]:
# Full detection test
!python test_detection.py

## Step 8: Run TracIn Influence Demo

In [ ]:
!python scripts/verify_model_setup.py --demo-tracin

## Optional: Check Results

View the detection output JSON.

In [ ]:
import json

results_file = 'detection_results.json'
if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)
    print(json.dumps(results, indent=2))
else:
    print("No detection_results.json found yet — run test_detection.py first")

---
## Notes

- Code lives at `/kaggle/working/Mini-Project/` (cloned from GitHub)
- Your fine-tuned checkpoints live at `/kaggle/input/<your-slug>/jie_checkpoints/` (from attached Kaggle Dataset)
- **Why Kaggle Dataset instead of GitHub?** `model.safetensors` for GPT-2 medium is ~1.5 GB — GitHub blocks files over 100 MB. Kaggle Datasets support up to 20 GB.
- Session data is wiped after ~9 hrs or on disconnect — download any output via `File → Download output`
- To re-run later: just re-run all cells top to bottom (Steps 1–4 take ~3 min)